# Reducción dimensional a 20 variables 

Para implementar con precisión la lógica que solicitas, aplicaremos un algoritmo clásico de **filtrado por correlación (Spearman)**. Este enfoque resolverá el problema de la siguiente manera:

1. **Alta correlación con el objetivo:** Evaluará la correlación de todas las variables contra `casos_dengue` y las ordenará de mayor a menor impacto.
2. **Baja correlación entre sí (Multicolinealidad):** Recorrerá las variables de forma descendente. Si una variable nueva está muy correlacionada con otra que ya seleccionamos anteriormente (por ejemplo, un coeficiente $> 0.7$), la descartará por "redundante". Esto evitará que el modelo reciba información duplicada (ruido).
3. **Las dos recomendaciones previas:** * Se entrena utilizando la variable logarítmica **`casos_ln`** como objetivo para suavizar los picos epidemiológicos.
* Al predecir, se aplica automáticamente la **transformación inversa `np.expm1()**` antes de calcular el MAE final y graficar, apuntando a bajar drásticamente el error en el testeo.



Aquí tienes el script de Python completo para realizar la selección dimensional exacta a 20 variables y entrenar el modelo ganador (XGBoost):

```python


El error se debe a un pequeño despiste en los *imports* de la cabecera (en la línea 3): el módulo `matplotlib.subplots` no existe como librería independiente para importar directamente mediante `import`.

En la siguiente línea (línea 4) ya estabas importando correctamente el módulo principal mediante `import matplotlib.pyplot as plt`, que es el que contiene a la función `subplots()`.

Para solucionarlo, simplemente debes **eliminar la línea 3**. Adicionalmente, corregí un detalle sintáctico en la línea 94 (`print(="*40)`), que tenía un error de asignación inválido al imprimir el separador visual.

Aquí tienes el script corregido y limpio listo para ejecutar:

```python


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt  # Importación correcta de matplotlib
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 1. Cargar los datos desde la nueva ubicación en Excel
ruta_archivo = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\2_spearman_2021_2025\2_data\1_raw\1_meteo_epi_rezagos.xlsx"
df = pd.read_excel(ruta_archivo)

# 2. Preprocesamiento e Índices
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)
df = df.sort_index()

# Crear la variable casos_ln en caso de que no exista en este archivo específico
if 'casos_ln' not in df.columns:
    df['casos_ln'] = np.log1p(df['casos_dengue'])

# 3. METODOLOGÍA DE REDUCCIÓN DIMENSIONAL (SPEARMAN)
target_original = 'casos_dengue'
target_log = 'casos_ln'

# Excluir identificadores y las variables objetivo de las características candidatas
excluir = [target_original, target_log, 'año', 'semana_epi']
candidatas = [col for col in df.columns if col not in excluir]

# Calcular matriz de correlación de Spearman completa
matriz_corr = df[[target_original] + candidatas].corr(method='spearman')

# Obtener la correlación absoluta con la variable objetivo y ordenar de mayor a menor
corr_con_objetivo = matriz_corr[target_original].drop(target_original).abs().sort_values(ascending=False)

# Algoritmo de filtrado: Seleccionar variables con alta correlación con el objetivo
# pero con baja correlación entre sí (umbral de multicolinealidad de 0.70)
variables_seleccionadas = []
umbral_multicolinealidad = 0.70

for var in corr_con_objetivo.index:
    if len(variables_seleccionadas) >= 20:  # Detenerse al alcanzar exactamente 20 variables
        break
        
    # Verificar si está muy correlacionada con alguna de las ya seleccionadas
    redundante = False
    for var_sel in variables_seleccionadas:
        if abs(matriz_corr.loc[var, var_sel]) > umbral_multicolinealidad:
            redundante = True
            break
            
    if not redundante:
        variables_seleccionadas.append(var)

print(f"=== REDUCCIÓN DIMENSIONAL COMPLETADA ===")
print(f"Se seleccionaron las mejores {len(variables_seleccionadas)} variables meteorológicas que evitan redundancia:\n")
for i, v in enumerate(variables_seleccionadas, 1):
    print(f"{i}. {v} (Corr Spearman: {corr_con_objetivo[v]:.4f})")
print("="*40)


=== REDUCCIÓN DIMENSIONAL COMPLETADA ===
Se seleccionaron las mejores 20 variables meteorológicas que evitan redundancia:

1. hum_esp_lag_6 (Corr Spearman: 0.5529)
2. hum_esp (Corr Spearman: 0.5107)
3. sst_lag_12 (Corr Spearman: 0.4796)
4. hum_rel_lag_8 (Corr Spearman: 0.4214)
5. hum_rel_lag_1 (Corr Spearman: 0.4072)
6. vel_vi_max_lag_4 (Corr Spearman: 0.3796)
7. vel_vi_max_lag_7 (Corr Spearman: 0.3533)
8. sst (Corr Spearman: 0.3447)
9. dias_lluvia_lag_5 (Corr Spearman: 0.3375)
10. vel_vi_max_lag_1 (Corr Spearman: 0.3365)
11. dias_lluvia_lag_4 (Corr Spearman: 0.3358)
12. dias_lluvia_lag_3 (Corr Spearman: 0.3346)
13. prec_lag_7 (Corr Spearman: 0.3272)
14. dias_lluvia_lag_2 (Corr Spearman: 0.3204)
15. prec_lag_6 (Corr Spearman: 0.3101)
16. vel_vi_lag_6 (Corr Spearman: 0.2987)
17. dias_lluvia (Corr Spearman: 0.2935)
18. soi_lag_12 (Corr Spearman: 0.2848)
19. prec_lag_1 (Corr Spearman: 0.2778)
20. soi_lag_11 (Corr Spearman: 0.2761)


In [7]:
df.to_excel(r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\2_spearman_2021_2025\2_data\2_processed\2_meteo_epi_rezagos_spearman_20.xlsx",index = True)

In [ ]:

# 4. Preparar matrices con las 20 variables seleccionadas
X = df[variables_seleccionadas]
y = df[target_log]  # Recomendación 1: Entrenar con el logaritmo

# 5. División temporal estricta de datos
X_train = X.loc['2021-01-01':'2025-09-30']
y_train = y.loc['2021-01-01':'2025-09-30']
X_test = X.loc['2025-10-01':'2025-12-31']
y_test = y.loc['2025-10-01':'2025-12-31']

# Guardar series reales en escala original (casos reales sin logaritmo) para evaluación final
y_train_original = df.loc['2021-01-01':'2025-09-30', target_original]
y_test_original = df.loc['2025-10-01':'2025-12-31', target_original]

# 6. Validación Cruzada Temporal y Afinamiento (Modelo Ganador anterior: XGBoost)
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 5, 6],
    'n_estimators': [100, 150, 200],
    'subsample': [0.7, 0.8, 0.9]
}

print("\nIniciando afinamiento de XGBoost con las 20 variables optimizadas...")
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, objective='reg:absoluteerror'),
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 7. Predicciones y RECOMENDACIÓN 2: Transformación Inversa (np.expm1)
y_train_pred_log = best_model.predict(X_train)
y_test_pred_log = best_model.predict(X_test)

y_train_pred_original = pd.Series(np.expm1(y_train_pred_log), index=y_train.index)
y_test_pred_original = pd.Series(np.expm1(y_test_pred_log), index=y_test.index)

# 8. Métricas Finales en la escala real de casos
mae_train_final = mean_absolute_error(y_train_original, y_train_pred_original)
mae_test_final = mean_absolute_error(y_test_original, y_test_pred_original)

print("\n" + "="*40)
print(f"RESULTADOS CON SELECCIÓN DE VARIABLES DE SPEARMAN:")
print(f"Mejores Parámetros: {grid_search.best_params_}")
print(f"MAE Entrenamiento Final: {mae_train_final:.4f}")
print(f"MAE Testeo Final (Q4 2025): {mae_test_final:.4f}")
print("="*40)

# 9. Gráficas Comparativas de Rendimiento
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Gráfica del periodo de entrenamiento
axes[0].plot(y_train_original.index, y_train_original.values, label='Real (Entrenamiento)', color='blue', alpha=0.5)
axes[0].plot(y_train_pred_original.index, y_train_pred_original.values, label='Predicho (XGBoost)', color='darkorange', linestyle='--')
axes[0].set_title(f'Rendimiento en el Entrenamiento (2021 - Q3 2025)\nMAE: {mae_train_final:.2f}')
axes[0].set_ylabel('Casos de Dengue')
axes[0].legend()
axes[0].grid(True)

# Gráfica del periodo de testeo
axes[1].plot(y_test_original.index, y_test_original.values, label='Real (Testeo Q4 2025)', color='green', marker='o')
axes[1].plot(y_test_pred_original.index, y_test_pred_original.values, label='Predicho (XGBoost)', color='red', linestyle='--', marker='x')
axes[1].set_title(f'Rendimiento en el Testeo (Q4 2025)\nMAE: {mae_test_final:.2f}')
axes[1].set_ylabel('Casos de Dengue')
axes[1].set_xlabel('Fecha')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

```

In [ ]:

# 4. Preparar matrices con las 20 variables seleccionadas
X = df[variables_seleccionadas]
y = df[target_log]  # Recomendación 1: Entrenar con el logaritmo

# 5. División temporal estricta de datos
X_train = X.loc['2021-01-01':'2025-09-30']
y_train = y.loc['2021-01-01':'2025-09-30']
X_test = X.loc['2025-10-01':'2025-12-31']
y_test = y.loc['2025-10-01':'2025-12-31']

# Guardar series reales en escala original (casos reales sin logaritmo) para evaluación final
y_train_original = df.loc['2021-01-01':'2025-09-30', target_original]
y_test_original = df.loc['2025-10-01':'2025-12-31', target_original]

# 6. Validación Cruzada Temporal y Afinamiento (Modelo Ganador anterior: XGBoost)
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 5, 6],
    'n_estimators': [100, 150, 200],
    'subsample': [0.7, 0.8, 0.9]
}

print("\nIniciando afinamiento de XGBoost con las 20 variables optimizadas...")
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, objective='reg:absoluteerror'),
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 7. Predicciones y RECOMENDACIÓN 2: Transformación Inversa (np.expm1)
y_train_pred_log = best_model.predict(X_train)
y_test_pred_log = best_model.predict(X_test)

y_train_pred_original = pd.Series(np.expm1(y_train_pred_log), index=y_train.index)
y_test_pred_original = pd.Series(np.expm1(y_test_pred_log), index=y_test.index)

# 8. Métricas Finales en la escala real de casos
mae_train_final = mean_absolute_error(y_train_original, y_train_pred_original)
mae_test_final = mean_absolute_error(y_test_original, y_test_pred_original)

print(f"\n" + "="*40)
print(f"RESULTADOS CON SELECCIÓN DE VARIABLES DE SPEARMAN:")
print(f"Mejores Parámetros: {grid_search.best_params_}")
print(f"MAE Entrenamiento Final: {mae_train_final:.4f}")
print(f"MAE Testeo Final (Q4 2025): {mae_test_final:.4f}")
print(="*40)

# 9. Gráficas Comparativas de Rendimiento
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Gráfica del periodo de entrenamiento
axes[0].plot(y_train_original.index, y_train_original.values, label='Real (Entrenamiento)', color='blue', alpha=0.5)
axes[0].plot(y_train_pred_original.index, y_train_pred_original.values, label='Predicho (XGBoost)', color='darkorange', linestyle='--')
axes[0].set_title(f'Rendimiento en el Entrenamiento (2021 - Q3 2025)\nMAE: {mae_train_final:.2f}')
axes[0].set_ylabel('Casos de Dengue')
axes[0].legend()
axes[0].grid(True)

# Gráfica del periodo de testeo
axes[1].plot(y_test_original.index, y_test_original.values, label='Real (Testeo Q4 2025)', color='green', marker='o')
axes[1].plot(y_test_pred_original.index, y_test_pred_original.values, label='Predicho (XGBoost)', color='red', linestyle='--', marker='x')
axes[1].set_title(f'Rendimiento en el Testeo (Q4 2025)\nMAE: {mae_test_final:.2f}')
axes[1].set_ylabel('Casos de Dengue')
axes[1].set_xlabel('Fecha')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()



```

### ¿Qué ventajas aporta este script?

* **Elimina el ruido:** El filtro descarta automáticamente variables meteorológicas que se comportan idéntico a otras (por ejemplo, evita meter `temp_lag_1` y `temp_lag_2` al mismo tiempo si aportan exactamente el mismo valor matemático).
* **Automatización:** Al imprimir la consola, el script te mostrará el listado explícito de las **20 variables ganadoras** ordenadas según su relevancia absoluta con el dengue, lo que te servirá directamente para la redacción de la sección de metodología en tu investigación.